# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities below are referenced by their `@id` fields as specified in the Croissant schema.

In [ ]:
# Explore the dataset object for available record sets
print("Available RecordSets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']} : {rs.get('name', 'Unnamed')}")

# For each RecordSet, show fields and columns
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet @id: {rs['@id']}, Name: {rs.get('name', 'Unnamed')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields (@id):")
        for field in fields:
            print(f"    - {field['@id']}: {field.get('name', 'Unnamed')} (dataType: {field.get('dataType', 'unknown')})")
    columns = rs.get('columns', [])
    if columns:
        print("  Columns (@id):")
        for col in columns:
            print(f"    - {col['@id']}: {col.get('name', 'Unnamed')}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

Below, we use `@id` references, for both record sets and fields, as instructed.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Extract data from each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For demonstration, show columns for first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we:
- Select a numeric field by its `@id` (e.g., GAD-7 total score)
- Filter records where GAD-7 > 10
- Normalize this field
- Group by a categorical field (e.g., `village`)

In [ ]:
# Choose the main record set for analysis
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# List columns to select numeric field
print("Available columns for EDA:")
print(df.columns.tolist())

# Let's pick the likely GAD-7 score field by typical survey naming
# Suppose '@id': 'gad_7_total' and 'village' fields are available.
numeric_field = 'gad_7_total' # Adjust as per actual ID
group_field = 'village' # Adjust as per actual ID

# Only continue EDA if these fields are available
if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field, group_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} for filtered records by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field}' not found in record set '{record_set_id}'. Please check the column list above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram (if field available)
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=20, edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# Boxplot by village/group if available
if group_field in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(12,5))
    df.boxplot(column=numeric_field, by=group_field, rot=90)
    plt.title(f"Boxplot of {numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey dataset provides granular insight into psychological health and demographics in Kilifi County, Kenya.
- Using the `mlcroissant` library with Croissant `@id` references enables transparent data workflows and proper provenance tracking.- High GAD-7 scores (above 10) were identified and grouped by village, demonstrating potential for targeted mental health interventions.
- Further analysis can extend to other fields, multi-field relationships, and integration with public health strategies.